# Parameters

In [ ]:
from common.utils import (
    check_initial_sector_omega_ratio,
    omega_c,
    delta0_from_N_Gamma,
    Omega0_from_N_Gamma,
    default_three_phase_protocol,
    validated_mcwf_dt,
)

from pathlib import Path
import numpy as np

%load_ext autoreload
%autoreload 2

seed = 1234
num_snapshots = 100

### Custom Quantum Trajectory - Single

In [ ]:
def run_single_trajectory(
          N=20,
          dN=0, 
          dt=1e-2,
          Gamma=1.0,
          sector_distribution="binomial",
          shifted_jump_operator = True,
        ):
    import time
    from quantum_trajectories.state_helpers import (
        centered_sector_initial_coeffs,
    )
    from quantum_trajectories.sim import simulate_single_trajectory
    from common.plotting import plot_trajectory_angles_and_excitation
    from quantum_trajectories.aggregator import (
        trajectory_observables,
        single_trajectory_to_averaged_result,
    )

    Omega_ratio = 0.4
    delta0 = delta0_from_N_Gamma(N, Gamma)

    # validate time step
    dt = validated_mcwf_dt(dt, N, Gamma)

    N_J = N // 2
    # Omega0 = Omega_ratio * omega_c(N_J, Gamma)
    Omega0 = Omega0_from_N_Gamma(N, Gamma)
    # Define the three-phase protocol with the desired parameters.
    phases = default_three_phase_protocol(
        T1=10.0,
        T2=10.0,
        T3=10.0,
        delta0=delta0,
        Omega0=Omega0,
    )

    # Returns a dictionary of key: sector Nj, value: coefficient for that sector.
    sector_coeffs = centered_sector_initial_coeffs(
        N, 
        dN, 
        sector_distribution=sector_distribution,
        )
    ratio_check = check_initial_sector_omega_ratio(
        sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid: "
            f"Omega={ratio_check['omega']}, Omega_c={ratio_check['omega_c']}, "
            f"smallest Nj={ratio_check['min_nj']}, ratio={ratio_check['ratio']}"
        )
        return None

    t0 = time.perf_counter()
    result = simulate_single_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        shifted_jump_operator=shifted_jump_operator,
    )

    simulation_runtime = time.perf_counter() - t0
    print(f"simulation runtime: {simulation_runtime:.3f} s")

    obs = trajectory_observables(result)
    obs = single_trajectory_to_averaged_result(result, obs)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        obs,
        phases,
        show_phase1_ss=True,
        output_path=output_dir / "qt_single.png",
        label="qt_single",
    )

run_single_trajectory(
    N=20,
    dN=0,
    sector_distribution="binomial",
    shifted_jump_operator=True,
)

### Custom Quantum Trajcetory - Ensamble

In [ ]:
def run_ensamble(
        N=20,
        dN=0,
        dt=1e-2,
        Gamma=1.0,
        sector_distribution="binomial",
        shifted_jump_operator=False,
        ntraj=100,
        n_processes=-1,
        ):
    import time

    from quantum_trajectories.state_helpers import (
        centered_sector_initial_coeffs,
    )
    from quantum_trajectories.ensamble_sim import run_trajectory_ensemble
    from quantum_trajectories.aggregator import (
        ensemble_observables,
        make_averaged_result
    )
    from common.plotting import plot_trajectory_angles_and_excitation
    from squeezing_parameter import plot_generalized_xi

    # Model and parameters
    Omega_ratio = 0.866
    # delta0 = delta0_from_N_Gamma(N, Gamma)
    delta0 = 1

    # dt = validated_mcwf_dt(dt, N, Gamma)

    N_J = N // 2
    Omega0 = Omega_ratio * omega_c(N_J, Gamma)
    # Omega0 = Omega0_from_N_Gamma(N, Gamma)

    # Define the three-phase protocol with the desired parameters.
    phases = default_three_phase_protocol(
        T1=5.0,
        T2=10.0,
        T3=5.0,
        delta0=delta0,
        Omega0=Omega0,
    )

    sector_coeffs = centered_sector_initial_coeffs(N, dN, sector_distribution=sector_distribution)
    ratio_check = check_initial_sector_omega_ratio(
        sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid: "
            f"Omega={ratio_check['omega']}, Omega_c={ratio_check['omega_c']}, "
            f"smallest Nj={ratio_check['min_nj']}, ratio={ratio_check['ratio']}"
        )
        return None

    t0 = time.perf_counter()
    ensemble = run_trajectory_ensemble(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        ntraj=ntraj,
        shifted_jump_operator=shifted_jump_operator,
        n_processes=n_processes,
        chunksize=1,
        verbose=True
    )
    simulation_runtime = time.perf_counter() - t0
    print(f"simulation runtime: {simulation_runtime:.3f} s")

    t0 = time.perf_counter()
    obs_avg = ensemble_observables(ensemble, n_processes=n_processes)
    print(f"ensemble_observables runtime: {time.perf_counter() - t0:.3f} s")

    t0 = time.perf_counter()
    avg_result = make_averaged_result(ensemble, obs_avg)
    print(f"make_averaged_result runtime: {time.perf_counter() - t0:.3f} s")

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        avg_result,
        phases,
        output_path=output_dir / "qt_ensable.png",
        show_phase1_ss=True,
        show_spread=False,
        label=f"ensamble run",
    )

    xi_data, xi_fig, xi_ax = plot_generalized_xi(
        ensemble,
        n_processes=n_processes,
        verbose=True,
        averaged_observables=obs_avg,
        output_path=output_dir / "qt_ensable_generalized_xi.png",
    )


run_ensamble(
    N=100, 
    dN=0, 
    dt=1e-2,
    Gamma=1.0,
    sector_distribution="binomial",
    shifted_jump_operator = True,
    ntraj=100,
    n_processes=-1
    )


### Custom Quantum Trajectory - Ensamble - half_width time scaling

In [ ]:
def run_ensamble_half_width_time_scaling(
    N,
    max_dN,
    dN_step,
    ntraj,
    dt=1e-2,
    Gamma=1.0,
    shifted_jump_operator=False,
    compute_theoretical_jump_stats=False,
    compute_theoretical_approx_jump_stats=False,
    verbose=False,
):
    import time
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator
    from tqdm.auto import tqdm

    from common.utils import default_three_phase_protocol, omega_c
    from quantum_trajectories.state_helpers import centered_sector_initial_coeffs
    from quantum_trajectories.ensamble_sim import run_trajectory_ensemble
    from quantum_trajectories.theory_validation import (
        theoretical_approx_jump_count_vs_half_width_data,
        theoretical_jump_rate_ensemble_summary,
    )

    print(f"Considering {N} atoms with max_dN={max_dN}")

    # Model and parameters
    dt = validated_mcwf_dt(dt, N, Gamma)
    Omega_ratio = 0.4
    delta0 = delta0_from_N_Gamma(N, Gamma)
    N_J = N // 2
    # Omega0 = Omega_ratio * omega_c(N_J, Gamma)
    Omega0 = Omega0_from_N_Gamma(N, Gamma)
    local_phases = default_three_phase_protocol(
        T1=10.0,
        T2=10.0,
        T3=10.0,
        delta0=delta0,
        Omega0=Omega0,
    )

    widest_dN_sector_coeffs = centered_sector_initial_coeffs(N, dN=max_dN, sector_distribution="square")
    ratio_check = check_initial_sector_omega_ratio(
        widest_dN_sector_coeffs,
        Omega=max(abs(phase.omega) for phase in local_phases),
        Gamma=Gamma,
    )
    if not ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid: "
            f"Omega={ratio_check['omega']}, Omega_c={ratio_check['omega_c']}, "
            f"smallest Nj={ratio_check['min_nj']}, ratio={ratio_check['ratio']}"
        )
        return None

    compare_phase2_counts = compute_theoretical_jump_stats or compute_theoretical_approx_jump_stats
    t_phase2_start = local_phases[0].duration
    t_phase2_end = local_phases[0].duration + local_phases[1].duration

    dN_values = list(range(0, max_dN + 1, dN_step))
    num_nj_sectors = []
    dN_runtimes = []
    avg_jump_counts = []
    avg_jump_count_sems = []
    theoretical_jump_counts = []
    theoretical_jump_count_sems = []
    theoretical_approx_jump_counts = []
    seen_num_nj_sectors = set()

    if compute_theoretical_approx_jump_stats:
        omega_over_omega_c = Omega0 / omega_c(N_J, Gamma)
        theoretical_approx_dN_data = theoretical_approx_jump_count_vs_half_width_data(
            delta=local_phases[1].delta,
            Gamma=Gamma,
            N=N,
            omega_over_omega_c=omega_over_omega_c,
            max_dN=max_dN,
            simulation_time=local_phases[1].duration,
        )

    for dN in tqdm(dN_values, desc="Half-width dN sweep"):
        sector_coeffs = centered_sector_initial_coeffs(N, dN=dN, sector_distribution="square")
        actual_num_nj_sectors = len(sector_coeffs)

        if actual_num_nj_sectors in seen_num_nj_sectors:
            continue
        seen_num_nj_sectors.add(actual_num_nj_sectors)

        t0 = time.perf_counter()
        ensemble = run_trajectory_ensemble(
            N=N,
            Gamma=Gamma,
            phases=local_phases,
            sector_coeffs=sector_coeffs,
            dt=dt,
            num_snapshots=num_snapshots,
            seed=seed,
            ntraj=ntraj,
            shifted_jump_operator=shifted_jump_operator,
            n_processes=-1,
            chunksize=1,
            verbose=verbose,
        )
        runtime = time.perf_counter() - t0
        if verbose:
            print(f"simulation runtime: {runtime:.3f} s")

        num_nj_sectors.append(actual_num_nj_sectors)
        dN_runtimes.append(runtime)
        if compare_phase2_counts:
            sim_jump_counts = np.asarray([
                sum(t_phase2_start <= t < t_phase2_end for t in traj.jump_times)
                for traj in ensemble.trajectories
            ], dtype=float)
        else:
            sim_jump_counts = np.asarray([traj.jump_count for traj in ensemble.trajectories], dtype=float)

        avg_jump_counts.append(float(np.mean(sim_jump_counts)))
        if sim_jump_counts.size > 1:
            avg_jump_count_sems.append(float(np.std(sim_jump_counts, ddof=1) / np.sqrt(sim_jump_counts.size)))
        else:
            avg_jump_count_sems.append(0.0)

        if compute_theoretical_jump_stats:
            theoretical_summary = theoretical_jump_rate_ensemble_summary(ensemble)
            theoretical_jump_counts.append(theoretical_summary["integrated_theoretical_jump_rate"]["mean"])
            if len(ensemble.trajectories) > 1:
                theoretical_jump_count_sems.append(
                    theoretical_summary["integrated_theoretical_jump_rate"]["std"] / np.sqrt(len(ensemble.trajectories))
                )
            else:
                theoretical_jump_count_sems.append(0.0)

        if compute_theoretical_approx_jump_stats:
            theoretical_approx_jump_counts.append(theoretical_approx_dN_data["jump_count"][dN])

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

    axes[0].plot(num_nj_sectors, dN_runtimes, marker="o", linewidth=1.8)
    axes[0].set_xlabel("number of Nj sectors")
    axes[0].set_ylabel("runtime (s)")
    axes[0].set_title(f"runtime vs number of Nj sectors (N={N})")
    axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[0].grid(alpha=0.3)
    axes[0].legend()

    jump_count_curves = []
    jump_count_handles = []
    jump_count_labels = []

    if compare_phase2_counts:
        sim_handle = axes[1].errorbar(num_nj_sectors, avg_jump_counts, yerr=avg_jump_count_sems, marker="o", linewidth=1.8, capsize=3, label=r"$\hat l = \hat J_- + i\frac{\Omega}{\Gamma}$")
        jump_count_handles.append(sim_handle)
        jump_count_labels.append(r"$\hat l = \hat J_- + i\frac{\Omega}{\Gamma}$")
        jump_count_curves.append(avg_jump_counts)
        jump_count_curves.append([y + err for y, err in zip(avg_jump_counts, avg_jump_count_sems)])
        axes[1].set_ylabel("average / expected phase-2 jump count")
        axes[1].set_title(f"average phase-2 jumps vs number of Nj sectors (N={N})")
    else:
        sim_handle = axes[1].errorbar(num_nj_sectors, avg_jump_counts, yerr=avg_jump_count_sems, marker="o", linewidth=1.8, capsize=3, label=r"$\hat l = \hat J_- + i\frac{\Omega}{\Gamma}$")
        jump_count_handles.append(sim_handle)
        jump_count_labels.append(r"$\hat l = \hat J_- + i\frac{\Omega}{\Gamma}$")
        jump_count_curves.append(avg_jump_counts)
        jump_count_curves.append([y + err for y, err in zip(avg_jump_counts, avg_jump_count_sems)])
        axes[1].set_ylabel("average jump count")
        axes[1].set_title(f"average jumps vs number of Nj sectors (N={N})")

    if compute_theoretical_jump_stats:
        theoretical_handle = axes[1].errorbar(num_nj_sectors, theoretical_jump_counts, yerr=theoretical_jump_count_sems, marker="o", linewidth=1.8, capsize=3, label=r"$\hat l \approx \frac{\delta}{\Gamma}\tan\tilde{\theta}_J + \frac{2\delta\sin\tilde{\theta}_J}{N\Gamma\cos^3\tilde{\theta}_J}\hat S_z$")
        jump_count_handles.append(theoretical_handle)
        jump_count_labels.append(r"$\hat l \approx \frac{\delta}{\Gamma}\tan\tilde{\theta}_J + \frac{2\delta\sin\tilde{\theta}_J}{N\Gamma\cos^3\tilde{\theta}_J}\hat S_z$")
        jump_count_curves.append(theoretical_jump_counts)
        jump_count_curves.append([y + err for y, err in zip(theoretical_jump_counts, theoretical_jump_count_sems)])
    if compute_theoretical_approx_jump_stats:
        (theoretical_approx_handle,) = axes[1].plot(num_nj_sectors, theoretical_approx_jump_counts, marker="o", linewidth=1.8, label=r"$\hat l \approx \frac{\delta}{\Gamma}\tan\tilde{\theta}_J + \frac{2\delta\sin\tilde{\theta}_J}{N\Gamma\cos^3\tilde{\theta}_J}\hat S_z,\; \hat S_z \approx \frac{N}{2} - \hat N_J$")
        jump_count_handles.append(theoretical_approx_handle)
        jump_count_labels.append(r"$\hat l \approx \frac{\delta}{\Gamma}\tan\tilde{\theta}_J + \frac{2\delta\sin\tilde{\theta}_J}{N\Gamma\cos^3\tilde{\theta}_J}\hat S_z,\; \hat S_z \approx \frac{N}{2} - \hat N_J$")
        jump_count_curves.append(theoretical_approx_jump_counts)

    axes[1].set_xlabel("number of Nj sectors")
    max_jump_count = max(max(curve) for curve in jump_count_curves if len(curve) > 0)
    axes[1].set_ylim(0.0, 1.1 * max_jump_count if max_jump_count > 0 else 1.0)
    axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))
    axes[1].grid(alpha=0.3)
    axes[1].legend(jump_count_handles, jump_count_labels)

    fig.tight_layout()
    fig.savefig(output_dir / f"qt_ensamble_half_width_time_scaling_N_{N}.png", dpi=200, bbox_inches="tight")

    # dN_runtimes, avg_jump_counts, theoretical_jump_counts, theoretical_approx_jump_counts



#std = int(np.sqrt(num_atoms)/2)

run_ensamble_half_width_time_scaling(
    N=1000,
    max_dN=80,
    dN_step=10,
    ntraj=100,
    dt=1e-2,
    Gamma=1.0,
    shifted_jump_operator=True,
    compute_theoretical_jump_stats=False,
    compute_theoretical_approx_jump_stats=False,
    verbose=True,
)


### Inhomogeneous Couplings

In [ ]:
def compare_homogeneous_vs_inhomogeneous(
    N=100,
    dN=0,
    N1=0,
    dt=1e-2,
    Gamma=1.0,
    ntraj=10,
    shifted_jump_operator=True,
    omega_1=1.0,
    n_processes=1,
):
    import time

    from quantum_trajectories.state_helpers import (
        centered_sector_initial_coeffs,
        centered_group_resolved_sector_initial_coeffs,
    )
    from quantum_trajectories.ensamble_sim import run_trajectory_ensemble
    from quantum_trajectories.aggregator import (
        ensemble_observables,
        make_averaged_result,
    )
    from common.plotting import plot_trajectory_angles_and_excitation
    from quantum_trajectories.inhomogeneous_diagnostics import (
        plot_inhomogeneous_group_angles,
        plot_inhomogeneous_mfe_residuals,
    )

    # Model and parameters
    Omega_ratio = 0.4
    delta0 = 1

    # dt = validated_mcwf_dt(dt, N, Gamma)

    N_J = N // 2
    N1 = N1
    N2 = N-N1
    Omega0 = Omega_ratio * omega_c(N_J, Gamma)
    # Omega0 = Omega0_from_N_Gamma(N, Gamma)

    phases = default_three_phase_protocol(
        T1=5.0,
        T2=10.0,
        T3=5.0,
        delta0=delta0,
        Omega0=Omega0,
    )

    homogeneous_sector_coeffs = centered_sector_initial_coeffs(
        N,
        dN=dN,
        sector_distribution="binomial",
    )
    inhomogeneous_sector_coeffs = centered_group_resolved_sector_initial_coeffs(
        N,
        dN=dN,
        N1=N1,
        N2=N2,
        sector_distribution="binomial",
    )
    print(
        "Using inhomogeneous validation setup with "
        f"N1={N1}, N2={N2}, omega_1={omega_1}. "
        f"Constructed sectors: {list(inhomogeneous_sector_coeffs.keys())}"
    )

    homogeneous_ratio_check = check_initial_sector_omega_ratio(
        homogeneous_sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not homogeneous_ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid for homogeneous run: "
            f"Omega={homogeneous_ratio_check['omega']}, Omega_c={homogeneous_ratio_check['omega_c']}, "
            f"smallest Nj={homogeneous_ratio_check['min_nj']}, ratio={homogeneous_ratio_check['ratio']}"
        )
        return None

    inhomogeneous_ratio_check = check_initial_sector_omega_ratio(
        inhomogeneous_sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not inhomogeneous_ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid for inhomogeneous run: "
            f"Omega={inhomogeneous_ratio_check['omega']}, Omega_c={inhomogeneous_ratio_check['omega_c']}, "
            f"smallest Nj={inhomogeneous_ratio_check['min_nj']}, ratio={inhomogeneous_ratio_check['ratio']}"
        )
        return None

    t0 = time.perf_counter()
    homogeneous_ensemble = run_trajectory_ensemble(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=homogeneous_sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        ntraj=ntraj,
        shifted_jump_operator=shifted_jump_operator,
        n_processes=n_processes,
        chunksize=1,
        verbose=True,
    )
    homogeneous_runtime = time.perf_counter() - t0
    homogeneous_avg = make_averaged_result(
        homogeneous_ensemble,
        ensemble_observables(homogeneous_ensemble, n_processes=n_processes),
    )

    t0 = time.perf_counter()
    inhomogeneous_ensemble = run_trajectory_ensemble(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=inhomogeneous_sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=2222,
        ntraj=ntraj,
        shifted_jump_operator=shifted_jump_operator,
        omega_1=omega_1,
        N1=N1,
        N2=N2,
        n_processes=n_processes,
        chunksize=1,
        verbose=True,
    )
    inhomogeneous_runtime = time.perf_counter() - t0
    inhomogeneous_avg = make_averaged_result(
        inhomogeneous_ensemble,
        ensemble_observables(inhomogeneous_ensemble, n_processes=n_processes),
    )

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        homogeneous_avg,
        phases,
        show_phase1_ss=True,
        label="homogeneous ensemble",
    )
    fig, axes = plot_trajectory_angles_and_excitation(
        inhomogeneous_avg,
        phases,
        axes=axes,
        show_phase1_ss=False,
        output_path=output_dir / "inhomogeneous_nj1_zero_vs_homogeneous.png",
        label=f"inhomogeneous ensemble (N1={N1}, omega_1={omega_1}",
    )
    plot_inhomogeneous_mfe_residuals(
        inhomogeneous_ensemble,
        omega1=omega_1,
        N1=N1,
        N2=N2,
        output_path=output_dir / "inhomogeneous_mfe_residuals.png",
    )
    plot_inhomogeneous_group_angles(
        inhomogeneous_ensemble,
        output_path=output_dir / "inhomogeneous_group_angles.png",
    )

    print(f"Homogeneous runtime: {homogeneous_runtime:.3f} s")
    print(f"Inhomogeneous runtime (N_J1=0): {inhomogeneous_runtime:.3f} s")
    if homogeneous_runtime > 0.0:
        print(f"Runtime ratio (inhom / hom): {inhomogeneous_runtime / homogeneous_runtime:.3f}")

    print(f"Homogeneous average jump count: {np.mean([traj.jump_count for traj in homogeneous_ensemble.trajectories]):.6f}")
    print(f"Inhomogeneous average jump count: {np.mean([traj.jump_count for traj in inhomogeneous_ensemble.trajectories]):.6f}")

    for key in ("Jx", "Jy", "Jz", "N_e", "jump_rate"):
        hom = np.asarray(getattr(homogeneous_avg.observables, key), dtype=float)
        inh = np.asarray(getattr(inhomogeneous_avg.observables, key), dtype=float)
        print(f"max |{key}_hom - {key}_inh| = {np.max(np.abs(hom - inh)):.6e}")


compare_homogeneous_vs_inhomogeneous(
    N=50,
    dN=2,
    N1=25,
    dt=1e-2,
    Gamma=1.0,
    ntraj=100,
    shifted_jump_operator=True,
    omega_1=0.75,
    n_processes=-1,
)


### Qutip Monte Carlo Solver

In [ ]:
def run_qutip_mcsolve(dt=1e-2, Gamma=1.0, shifted_jump_operator=False):
    from quantum_trajectories_qutip.sim import (
        simulate_fixed_nj_mc_trajectory
    )
    from quantum_trajectories_qutip.aggregator import qutip_fixed_nj_mcsolve_observables
    from common.plotting import plot_trajectory_angles_and_excitation

    dt = validated_mcwf_dt(dt, N, Gamma)

    qt_sim = simulate_fixed_nj_mc_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        num_points=600,
        ntraj=1,
        seed=1234,
        shifted_jump_operator=shifted_jump_operator,
        n_processes=None,
    )

    obs_qutip = qutip_fixed_nj_mcsolve_observables(qt_sim)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        obs_qutip,
        phases,
        output_path=output_dir / "qt_qutip.png",
        axes=None,
        label="qt_qutip",

    )

run_qutip_mcsolve(dt=1e-2, Gamma=1.0, shifted_jump_operator = False)

### Qutip Master Equation Solver

In [ ]:
def run_qutip_mesolve(dt=1e-2, Gamma=1.0, shifted_jump_operator=False):
    from quantum_trajectories_qutip.sim import simulate_fixed_nj_me_trajectory
    from quantum_trajectories_qutip.aggregator import qutip_fixed_nj_mcsolve_observables
    from common.plotting import plot_trajectory_angles_and_excitation

    dt = validated_mcwf_dt(dt, N, Gamma)

    me_res = simulate_fixed_nj_me_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        num_points=600,
        shifted_jump_operator=shifted_jump_operator,
    )

    me_obs = qutip_fixed_nj_mcsolve_observables(me_res)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        me_obs,
        phases,
        output_path=output_dir / "qt_qutip.png",
        axes=None,
        label="qt_qutip",

    )

run_qutip_mesolve(dt=1e-2, Gamma=1.0, shifted_jump_operator = False)

##### Single Quantum Trajectories - Custom vs Qutip

In [ ]:
def mcsolve_v_custom_trajectory(dt=1e-2, Gamma=1.0, print_numerics = False, shifted_jump_operator=False):
    from quantum_trajectories.state_helpers import centered_sector_initial_coeffs
    from quantum_trajectories.sim import simulate_single_trajectory
    from common.plotting import plot_trajectory_angles_and_excitation
    from quantum_trajectories.aggregator import (
        trajectory_observables,
        single_trajectory_to_averaged_result,
    )
    from quantum_trajectories_qutip.sim import simulate_fixed_nj_mc_trajectory
    from quantum_trajectories_qutip.aggregator import qutip_fixed_nj_mcsolve_observables

    dt = validated_mcwf_dt(dt, N, Gamma)

    # Custom single trajectory
    sector_coeffs = centered_sector_initial_coeffs(N, dN=0, sector_distribution="square")
    ratio_check = check_initial_sector_omega_ratio(
        sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid: "
            f"Omega={ratio_check['omega']}, Omega_c={ratio_check['omega_c']}, "
            f"smallest Nj={ratio_check['min_nj']}, ratio={ratio_check['ratio']}"
        )
        return None

    result = simulate_single_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        shifted_jump_operator=shifted_jump_operator,
    )
    obs = trajectory_observables(result)
    obs = single_trajectory_to_averaged_result(result, obs)

    # Qutip single trajectory
    qt_sim = simulate_fixed_nj_mc_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        num_points=600,
        ntraj=1,
        seed=seed,
        shifted_jump_operator=shifted_jump_operator,
    )
    obs_qutip = qutip_fixed_nj_mcsolve_observables(qt_sim)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        obs,
        phases,
        label="custom trajectory",
    )
    fig, axes = plot_trajectory_angles_and_excitation(
        obs_qutip,
        phases,
        axes=axes,
        label="qutip trajectory",
        show_phase1_ss=True,
        output_path=output_dir / "qutip_vs_custom.png",
    )

    if print_numerics:
        traj_jump_times = np.round(np.array(result.jump_times), 4)
        qutip_traj_times = np.round(np.array(qt_sim["result"].col_times[0]), 4)

        print("traj_jump_times", traj_jump_times)
        print("qutip_traj_times", qutip_traj_times)

        qutip_traj_jumps = qt_sim["result"].col_which
        print("qutip_traj_jumps", qutip_traj_jumps)


mcsolve_v_custom_trajectory(dt=1e-2, Gamma=1.0, shifted_jump_operator=False)


In [ ]:
def observable_mse(
    ntraj=10,
    qutip_num_points=600,
    dt=1e-3,
    Gamma=1.0,
    *,
    run_custom=True,
    run_mcsolve=True,
    run_mesolve=True,
    shifted_jump_operator=False,
):
    import time
    import numpy as np

    from common.plotting import plot_mse_vs_time
    from common.utils import observable_mse_by_time
    from quantum_trajectories_qutip.sim import (
        simulate_fixed_nj_mc_trajectory,
        simulate_fixed_nj_me_trajectory,
    )
    from quantum_trajectories_qutip.aggregator import qutip_fixed_nj_mcsolve_observables
    from quantum_trajectories.aggregator import (
        ensemble_observables,
        make_averaged_result,
    )
    from quantum_trajectories.state_helpers import centered_sector_initial_coeffs
    from quantum_trajectories.ensamble_sim import run_trajectory_ensemble

    dt = validated_mcwf_dt(dt, N, Gamma)

    keys = ("Jx", "Jy", "Jz", "N_e")
    cache = globals().setdefault("_observable_mse_cache", {})

    phase_signature = tuple(
        (float(p.duration), float(p.omega), float(p.delta), str(p.label))
        for p in phases
    )

    signatures = {
        "custom": (N, Gamma, phase_signature, dt, num_snapshots, seed, ntraj, shifted_jump_operator),
        "mcsolve": (N, Gamma, phase_signature, qutip_num_points, seed, ntraj, shifted_jump_operator),
        "mesolve": (N, Gamma, phase_signature, 10 * qutip_num_points, shifted_jump_operator),
    }

    def get_cached(name, run_now):
        cached = cache.get(name)
        if run_now:
            return None
        if cached is None:
            raise ValueError(
                f"No cached {name} result is available. Rerun with run_{name}=True."
            )
        if cached["signature"] != signatures[name]:
            raise ValueError(
                f"Cached {name} result was built with different parameters. "
                f"Rerun with run_{name}=True."
            )
        return cached["result"], cached.get("runtime_seconds")

    runtimes = {}

    if run_custom:
        sector_coeffs = centered_sector_initial_coeffs(N, dN=0, sector_distribution="square")
        t0 = time.perf_counter()
        res_ensemble = run_trajectory_ensemble(
            N=N,
            Gamma=Gamma,
            phases=phases,
            sector_coeffs=sector_coeffs,
            dt=dt,
            num_snapshots=num_snapshots,
            seed=seed,
            ntraj=ntraj,
            shifted_jump_operator=shifted_jump_operator,
        )
        runtimes["custom"] = time.perf_counter() - t0
        obs_custom = make_averaged_result(
            res_ensemble,
            ensemble_observables(res_ensemble),
        )
        cache["custom"] = {
            "signature": signatures["custom"],
            "result": obs_custom,
            "runtime_seconds": runtimes["custom"],
        }
    else:
        obs_custom, runtimes["custom"] = get_cached("custom", run_custom)

    if run_mcsolve:
        t0 = time.perf_counter()
        mc_res = simulate_fixed_nj_mc_trajectory(
            N=N,
            Gamma=Gamma,
            phases=phases,
            num_points=qutip_num_points,
            ntraj=ntraj,
            seed=seed,
            shifted_jump_operator=shifted_jump_operator,
        )
        runtimes["mcsolve"] = time.perf_counter() - t0
        obs_mc = qutip_fixed_nj_mcsolve_observables(mc_res)
        cache["mcsolve"] = {
            "signature": signatures["mcsolve"],
            "result": obs_mc,
            "runtime_seconds": runtimes["mcsolve"],
        }
    else:
        obs_mc, runtimes["mcsolve"] = get_cached("mcsolve", run_mcsolve)

    if run_mesolve:
        t0 = time.perf_counter()
        me_res = simulate_fixed_nj_me_trajectory(
            N=N,
            Gamma=Gamma,
            phases=phases,
            num_points=10 * qutip_num_points,
            shifted_jump_operator=shifted_jump_operator,
        )
        runtimes["mesolve"] = time.perf_counter() - t0
        obs_me = qutip_fixed_nj_mcsolve_observables(me_res)
        cache["mesolve"] = {
            "signature": signatures["mesolve"],
            "result": obs_me,
            "runtime_seconds": runtimes["mesolve"],
        }
    else:
        obs_me, runtimes["mesolve"] = get_cached("mesolve", run_mesolve)

    mse_custom = observable_mse_by_time(obs_custom, obs_me, keys=keys)
    mse_mcsolve = observable_mse_by_time(obs_mc, obs_me, keys=keys)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    plot_mse_vs_time(
        {
            "custom trajectories": mse_custom,
            "qutip trajectories": mse_mcsolve,
        },
        keys=keys,
        output_path=output_dir / "accuracy_qutip_vs_custom.png",
    )

    summary = {}
    for label, mse_data in {
        "custom_vs_mesolve": mse_custom,
        "mcsolve_vs_mesolve": mse_mcsolve,
    }.items():
        out = {
            key: {
                "mean_mse": mse_data[key]["mean_mse"],
                "integrated_mse": mse_data[key]["integrated_mse"],
            }
            for key in keys
        }
        out["score"] = float(np.mean([
            np.sqrt(out["Jx"]["integrated_mse"]),
            np.sqrt(out["Jy"]["integrated_mse"]),
            np.sqrt(out["Jz"]["integrated_mse"]),
            np.sqrt(out["N_e"]["integrated_mse"]),
        ]))
        summary[label] = out

    print(f"MSE custom: {summary['custom_vs_mesolve']['score']}")
    print(f"MSE mcsolve: {summary['mcsolve_vs_mesolve']['score']}")
    print(f"Runtime custom: {runtimes['custom']:.3f} s")
    print(f"Runtime mcsolve: {runtimes['mcsolve']:.3f} s")
    print(f"Runtime mesolve: {runtimes['mesolve']:.3f} s")


# Examples:
observable_mse(ntraj=100, dt=1e-2, Gamma=1.0, run_custom=True, run_mcsolve=True, run_mesolve=True, shifted_jump_operator=True)


In [ ]:
def compare_shifted_jump_operator_single_trajectory(dt=1e-2, Gamma=1.0):
    from quantum_trajectories.state_helpers import centered_sector_initial_coeffs
    from quantum_trajectories.sim import simulate_single_trajectory
    from common.plotting import plot_trajectory_angles_and_excitation
    from quantum_trajectories.aggregator import (
        trajectory_observables,
        single_trajectory_to_averaged_result,
    )

    dt = validated_mcwf_dt(dt, N, Gamma)

    sector_coeffs = centered_sector_initial_coeffs(N, dN=0, sector_distribution="square")
    ratio_check = check_initial_sector_omega_ratio(
        sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid: "
            f"Omega={ratio_check['omega']}, Omega_c={ratio_check['omega_c']}, "
            f"smallest Nj={ratio_check['min_nj']}, ratio={ratio_check['ratio']}"
        )
        return None

    result_true = simulate_single_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        shifted_jump_operator=True,
    )
    obs_true = trajectory_observables(result_true)
    obs_true = single_trajectory_to_averaged_result(result_true, obs_true)

    result_false = simulate_single_trajectory(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        shifted_jump_operator=False,
    )
    obs_false = trajectory_observables(result_false)
    obs_false = single_trajectory_to_averaged_result(result_false, obs_false)

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        obs_false,
        phases,
        label="regular jump operator",
    )
    fig, axes = plot_trajectory_angles_and_excitation(
        obs_true,
        phases,
        axes=axes,
        output_path=output_dir / "shifted_jump_operator.png",
        label="shifted jump operator",
    )

    print(f"Jump count regular jump operator: {result_false.jump_count}")
    print(f"Jump count shifted jump operator: {result_true.jump_count}")

compare_shifted_jump_operator_single_trajectory(dt=1e-2, Gamma=1.0)


In [ ]:
def compare_shifted_jump_operator_ensemble(dt=1e-2, Gamma=1.0):
    import time
    from quantum_trajectories.state_helpers import centered_sector_initial_coeffs
    from quantum_trajectories.ensamble_sim import run_trajectory_ensemble
    from quantum_trajectories.aggregator import (
        ensemble_observables,
        make_averaged_result,
    )
    from common.plotting import plot_trajectory_angles_and_excitation

    dt = validated_mcwf_dt(dt, N, Gamma)

    sector_coeffs = centered_sector_initial_coeffs(N, dN=0, sector_distribution="square")
    ratio_check = check_initial_sector_omega_ratio(
        sector_coeffs,
        Omega=max(abs(phase.omega) for phase in phases),
        Gamma=Gamma,
    )
    if not ratio_check["is_valid"]:
        print(
            "Omega/Omega_c check not valid: "
            f"Omega={ratio_check['omega']}, Omega_c={ratio_check['omega_c']}, "
            f"smallest Nj={ratio_check['min_nj']}, ratio={ratio_check['ratio']}"
        )
        return None

    t0 = time.perf_counter()
    ensemble_true = run_trajectory_ensemble(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        ntraj=100,
        shifted_jump_operator=True,
    )
    runtime_true = time.perf_counter() - t0
    obs_true = make_averaged_result(
        ensemble_true,
        ensemble_observables(ensemble_true),
    )

    t0 = time.perf_counter()
    ensemble_false = run_trajectory_ensemble(
        N=N,
        Gamma=Gamma,
        phases=phases,
        sector_coeffs=sector_coeffs,
        dt=dt,
        num_snapshots=num_snapshots,
        seed=seed,
        ntraj=100,
        shifted_jump_operator=False,
    )
    runtime_false = time.perf_counter() - t0
    obs_false = make_averaged_result(
        ensemble_false,
        ensemble_observables(ensemble_false),
    )

    output_dir = Path("output")
    output_dir.mkdir(exist_ok=True)
    fig, axes = plot_trajectory_angles_and_excitation(
        obs_false,
        phases,
        label="regular jump operator ensemble",
    )
    fig, axes = plot_trajectory_angles_and_excitation(
        obs_true,
        phases,
        axes=axes,
        output_path=output_dir / "shifted_jump_operator_ensemble.png",
        label="shifted jump operator ensemble",
    )

    print(f"Runtime regular jump operator ensemble: {runtime_false:.3f} s")
    print(f"Runtime shifted jump operator ensemble: {runtime_true:.3f} s")

compare_shifted_jump_operator_ensemble(dt=1e-2, Gamma=1.0)


In [ ]:
# from squeezing_parameter import plot_generalized_xi

# %load_ext autoreload
# %autoreload 2

# # Steady state angles
# theta_ss, phi_ss = phase1_ss_angles_for_nj(N_J, Omega0, Gamma)

# # tilde_theta, tilde_phi are the phase-1 reference angles used to define |1>
# xi_data, fig, ax = plot_generalized_xi(
#     result,
#     phases,
#     tilde_theta=theta_ss,
#     tilde_phi=phi_ss,
#     output_path="generalized_xi.png",
# )

import sys, numpy
print(sys.executable)
print(numpy.__version__)